# Flickr8k Data Exploration

Quick look at the dataset before building the preprocessing pipeline:
how many images/captions there are, what a raw vs. cleaned caption looks
like, and the caption-length distribution (which determines `max_length`
for padding).

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.data_preprocessing import load_raw_captions, clean_captions_mapping, get_splits

raw_captions = load_raw_captions()
print("Number of images with captions:", len(raw_captions))

sample_id = list(raw_captions.keys())[0]
print("\nSample image:", sample_id)
for c in raw_captions[sample_id]:
    print(" -", c)

In [ ]:
cleaned = clean_captions_mapping(raw_captions)
print("Cleaned captions for the same image:")
for c in cleaned[sample_id]:
    print(" -", c)

In [ ]:
train_ids, val_ids, test_ids = get_splits()
print(f"Train images: {len(train_ids)}")
print(f"Validation images: {len(val_ids)}")
print(f"Test images: {len(test_ids)}")
print(f"Total: {len(train_ids) + len(val_ids) + len(test_ids)}")

# Confirm there is no overlap (no image leakage) between splits.
assert not (set(train_ids) & set(val_ids))
assert not (set(train_ids) & set(test_ids))
assert not (set(val_ids) & set(test_ids))
print("No image leakage between splits: OK")

In [ ]:
import matplotlib.pyplot as plt

lengths = [len(c.split()) for caps in cleaned.values() for c in caps]
plt.figure(figsize=(6, 4))
plt.hist(lengths, bins=30)
plt.xlabel("Caption length (words, incl. startseq/endseq)")
plt.ylabel("Count")
plt.title("Distribution of caption lengths")
plt.show()

print("Max length:", max(lengths))
print("Mean length: %.2f" % (sum(lengths) / len(lengths)))

In [ ]:
from PIL import Image
from src import config

img_path = os.path.join(config.IMAGES_DIR, sample_id)
img = Image.open(img_path)
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis("off")
plt.title(cleaned[sample_id][0])
plt.show()

## Why this project is restricted to dogs and cats

An earlier, unrestricted version of this project trained on all 8,091
Flickr8k images. It kept defaulting to "a dog is running through the
grass" for images it was unsure about -- including real photos of cats.
The cell below is the actual analysis that explained why: Flickr8k's
animal photos are extremely dominated by dogs.

In [ ]:
from src.data_preprocessing import classify_dog_cat

dog_only, cat_only, both, neither = [], [], [], []
for image_id, caps in raw_captions.items():
    is_dog, is_cat = classify_dog_cat(caps)
    if is_dog and is_cat:
        both.append(image_id)
    elif is_dog:
        dog_only.append(image_id)
    elif is_cat:
        cat_only.append(image_id)
    else:
        neither.append(image_id)

print("dog-only images:", len(dog_only))
print("cat-only images:", len(cat_only))
print("both dog+cat images:", len(both))
print("neither:", len(neither))
print("total:", len(raw_captions))
print()
print(f"Ratio of dog-only to cat-only images: {len(dog_only) / len(cat_only):.0f}:1")

Given that ~200:1 imbalance, the project narrows scope to dogs and cats
only, caps dogs at 300 images, keeps all cat images, and oversamples
cats during training. See `build_dog_cat_splits` in
`src/data_preprocessing.py` and the main README's Dataset section for
the full reasoning and the resulting train/val/test sizes.

In [ ]:
from src.data_preprocessing import build_dog_cat_splits

dc_train, dc_val, dc_test, stats = build_dog_cat_splits(raw_captions)
print(stats)
print(f"Total dog/cat images used: {len(dc_train) + len(dc_val) + len(dc_test)}")